In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
print(os.listdir('/kaggle/input/banmani/'))

In [ ]:
import os
print(os.listdir('/kaggle/input/'))

In [ ]:
import os
print(os.listdir('/kaggle/input/datasets/'))

In [ ]:
import os
print(os.listdir('/kaggle/input/datasets/saikasatter/'))

In [ ]:
import os
print(os.listdir('/kaggle/input/datasets/saikasatter/banmani/'))

In [ ]:
import os, shutil
os.makedirs('/kaggle/working/data', exist_ok=True)
shutil.copy('/kaggle/input/datasets/saikasatter/banmani/preprocess.py', '/kaggle/working/data/preprocess.py')
shutil.copy('/kaggle/input/datasets/saikasatter/banmani/train_fixed.py', '/kaggle/working/train_fixed.py')
shutil.copy('/kaggle/input/datasets/saikasatter/banmani/BanMANI.csv', '/kaggle/working/BanMANI.csv')
print("Files copied successfully")
print(os.listdir('/kaggle/working/'))


In [ ]:
!pip install transformers==4.40.0 --quiet

In [ ]:
!python /kaggle/working/train_fixed.py sagorsarker/bangla-bert-base /kaggle/working/BanMANI.csv

In [ ]:
%%writefile /kaggle/working/train_fixed.py
import sys, os, json
import numpy as np
import pandas as pd
sys.path.insert(0, '/kaggle/working')
from data.preprocess import load_dataset
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

class BanMANIDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.encodings = tokenizer(texts, max_length=max_len, truncation=True, padding=False)
        self.labels = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), d

In [ ]:
!python /kaggle/working/train_fixed.py sagorsarker/bangla-bert-base /kaggle/working/BanMANI.csv

In [ ]:
code = '''import sys, os, json
import numpy as np
import pandas as pd
sys.path.insert(0, "/kaggle/working")
from data.preprocess import load_dataset
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

class BanMANIDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.encodings = tokenizer(texts, max_length=max_len, truncation=True, padding=False)
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_mani": f1_score(labels, preds, pos_label=1),
        "roc_auc": roc_auc_score(labels, probs[:,1]),
    }

def train(model_name, csv_path, epochs=5):
    print("Fine-tuning:", model_name)
    train_df, test_df = load_dataset(csv_path)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, ignore_mismatched_sizes=True)
    train_ds = BanMANIDataset(train_df["text_sep_token"].tolist(), train_df["label"].tolist(), tokenizer)
    test_ds = BanMANIDataset(test_df["text_sep_token"].tolist(), test_df["label"].tolist(), tokenizer)
    args = TrainingArguments(
        output_dir="./output",
        num_train_epochs=epochs,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=4,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_steps=20,
        report_to="none",
        seed=42)
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics)
    trainer.train()
    results = trainer.evaluate()
    print("FINAL RESULTS:", json.dumps({k:round(v,4) for k,v in results.items() if "runtime" not in k}, indent=2))
    return results

if __name__ == "__main__":
    model_name = sys.argv[1] if len(sys.argv) > 1 else "xlm-roberta-base"
    csv = sys.argv[2] if len(sys.argv) > 2 else "/kaggle/working/BanMANI.csv"
    train(model_name, csv)
'''

with open('/kaggle/working/train_fixed.py', 'w') as f:
    f.write(code)

print("File written successfully")
print("Lines:", len(code.splitlines()))

In [ ]:
!python /kaggle/working/train_fixed.py sagorsarker/bangla-bert-base /kaggle/working/BanMANI.csv

In [ ]:
!pip install transformers==4.44.0 peft==0.12.0 --quiet

In [1]:
import os, shutil
os.makedirs('/kaggle/working/data', exist_ok=True)
shutil.copy('/kaggle/input/datasets/saikasatter/banmani/preprocess.py', '/kaggle/working/data/preprocess.py')
shutil.copy('/kaggle/input/datasets/saikasatter/banmani/BanMANI.csv', '/kaggle/working/BanMANI.csv')

'/kaggle/working/BanMANI.csv'

In [2]:
code = '''import sys, os, json
import numpy as np
import pandas as pd
sys.path.insert(0, "/kaggle/working")
from data.preprocess import load_dataset
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

class BanMANIDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.encodings = tokenizer(texts, max_length=max_len, truncation=True, padding=False)
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1).numpy()
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_mani": f1_score(labels, preds, pos_label=1),
        "roc_auc": roc_auc_score(labels, probs[:,1]),
    }

def train(model_name, csv_path, epochs=5):
    print("Fine-tuning:", model_name)
    train_df, test_df = load_dataset(csv_path)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, ignore_mismatched_sizes=True)
    train_ds = BanMANIDataset(train_df["text_sep_token"].tolist(), train_df["label"].tolist(), tokenizer)
    test_ds = BanMANIDataset(test_df["text_sep_token"].tolist(), test_df["label"].tolist(), tokenizer)
    args = TrainingArguments(
        output_dir="./output",
        num_train_epochs=epochs,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=4,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_steps=20,
        report_to="none",
        seed=42)
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics)
    trainer.train()
    results = trainer.evaluate()
    print("FINAL RESULTS:", json.dumps({k:round(v,4) for k,v in results.items() if "runtime" not in k}, indent=2))
    return results

if __name__ == "__main__":
    model_name = sys.argv[1] if len(sys.argv) > 1 else "xlm-roberta-base"
    csv = sys.argv[2] if len(sys.argv) > 2 else "/kaggle/working/BanMANI.csv"
    train(model_name, csv)
'''

with open('/kaggle/working/train_fixed.py', 'w') as f:
    f.write(code)
print("Done")

Done


In [3]:
!python /kaggle/working/train_fixed.py sagorsarker/bangla-bert-base /kaggle/working/BanMANI.csv

2026-05-23 16:12:07.123983: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779552727.147361     225 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779552727.155530     225 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779552727.175215     225 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779552727.175275     225 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779552727.175283     225 computation_placer.cc:177] computation placer alr

In [4]:
!python /kaggle/working/train_fixed.py xlm-roberta-base /kaggle/working/BanMANI.csv

2026-05-23 16:20:30.530291: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779553230.555081    1249 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779553230.562464    1249 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779553230.583345    1249 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779553230.583375    1249 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779553230.583379    1249 computation_placer.cc:177] computation placer alr

In [5]:
!python /kaggle/working/train_fixed.py bert-base-multilingual-cased /kaggle/working/BanMANI.csv

2026-05-23 16:42:04.022914: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779554524.046433    2272 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779554524.054377    2272 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779554524.075083    2272 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779554524.075109    2272 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779554524.075113    2272 computation_placer.cc:177] computation placer alr

In [6]:
import json, pandas as pd

df = pd.DataFrame([
    {
        "Model": "TF-IDF word + LR",
        "Type": "Classical ML",
        "Accuracy": 0.5867,
        "F1 Macro": 0.5717,
        "F1 MANI": 0.6517,
        "ROC-AUC": 0.7018,
    },
    {
        "Model": "TF-IDF char + LR",
        "Type": "Classical ML",
        "Accuracy": 0.5733,
        "F1 Macro": 0.5656,
        "F1 MANI": 0.6235,
        "ROC-AUC": 0.6629,
    },
    {
        "Model": "TF-IDF + SVM",
        "Type": "Classical ML",
        "Accuracy": 0.5733,
        "F1 Macro": 0.5579,
        "F1 MANI": 0.6404,
        "ROC-AUC": "N/A",
    },
    {
        "Model": "mBERT (fine-tuned)",
        "Type": "Transformer",
        "Accuracy": 0.4067,
        "F1 Macro": 0.3535,
        "F1 MANI": 0.5389,
        "ROC-AUC": 0.5021,
    },
    {
        "Model": "XLM-RoBERTa (fine-tuned)",
        "Type": "Transformer",
        "Accuracy": 0.5933,
        "F1 Macro": 0.3724,
        "F1 MANI": 0.0,
        "ROC-AUC": 0.4896,
    },
    {
        "Model": "BanglaBERT (fine-tuned)",
        "Type": "Transformer",
        "Accuracy": 0.4667,
        "F1 Macro": 0.4258,
        "F1 MANI": 0.5789,
        "ROC-AUC": 0.5791,
    },
])

# Print table
print(df.to_string(index=False))

# Save CSV
df.to_csv('/kaggle/working/banmani_results.csv', index=False)

# Save JSON
with open('/kaggle/working/banmani_results.json', 'w') as f:
    json.dump(df.to_dict(orient='records'), f, indent=2)

print("\nSaved: banmani_results.csv and banmani_results.json")

                   Model         Type  Accuracy  F1 Macro  F1 MANI ROC-AUC
        TF-IDF word + LR Classical ML    0.5867    0.5717   0.6517  0.7018
        TF-IDF char + LR Classical ML    0.5733    0.5656   0.6235  0.6629
            TF-IDF + SVM Classical ML    0.5733    0.5579   0.6404     N/A
      mBERT (fine-tuned)  Transformer    0.4067    0.3535   0.5389  0.5021
XLM-RoBERTa (fine-tuned)  Transformer    0.5933    0.3724   0.0000  0.4896
 BanglaBERT (fine-tuned)  Transformer    0.4667    0.4258   0.5789  0.5791

Saved: banmani_results.csv and banmani_results.json


In [7]:
from IPython.display import FileLink
display(FileLink('/kaggle/working/banmani_results.csv'))
display(FileLink('/kaggle/working/banmani_results.json'))

/kaggle/working/banmani_results.csv

/kaggle/working/banmani_results.json

In [8]:
from IPython.display import FileLink
display(FileLink('/kaggle/working/banmani_results.csv'))
display(FileLink('/kaggle/working/banmani_results.json'))

/kaggle/working/banmani_results.csv

/kaggle/working/banmani_results.json

In [1]:
import torch, os
print("GPU:", torch.cuda.is_available())
print("Files:", os.listdir('/kaggle/working/'))

GPU: True
Files: ['.virtual_documents']


In [8]:
import os, shutil
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
print("Token loaded:", hf_token[:10], "...")

os.makedirs('/kaggle/working/data', exist_ok=True)
shutil.copy('/kaggle/input/datasets/saikasatter/banmani/preprocess.py',
            '/kaggle/working/data/preprocess.py')
shutil.copy('/kaggle/input/datasets/saikasatter/banmani/BanMANI.csv',
            '/kaggle/working/BanMANI.csv')
print("Files copied")
print("Working dir:", os.listdir('/kaggle/working/'))

Token loaded: hf_LdAIBuG ...
Files copied
Working dir: ['.virtual_documents', 'BanMANI.csv', 'data']


In [9]:
!pip install transformers==4.44.0 accelerate bitsandbytes -q
print("Install done")

Install done


In [10]:
code = '''import sys, os, json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
sys.path.insert(0, "/kaggle/working")
from data.preprocess import load_dataset
import torch
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

class BanMANIDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.encodings = tokenizer(texts, max_length=max_len,
                                   truncation=True, padding=False)
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_mani": f1_score(labels, preds, pos_label=1, zero_division=0),
    }

def run_seed(csv_path, seed, model_name="sagorsarker/bangla-bert-base", epochs=5):
    print(f"\\n{\\'=\\'*55}")
    print(f"  BanglaBERT | Re-stratified split | Seed {seed}")
    print(f"{\\'=\\'*55}")

    train_df, test_df = load_dataset(csv_path)
    combined = pd.concat([train_df, test_df], ignore_index=True)
    tr, te = train_test_split(combined, test_size=150,
                               stratify=combined["label"], random_state=seed)
    tr = tr.reset_index(drop=True)
    te = te.reset_index(drop=True)

    print(f"  Train: {len(tr)} samples ({tr[\\'label\\'].mean()*100:.1f}% MANI)")
    print(f"  Test:  {len(te)} samples ({te[\\'label\\'].mean()*100:.1f}% MANI)")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, ignore_mismatched_sizes=True)

    train_ds = BanMANIDataset(tr["text_sep_token"].tolist(),
                               tr["label"].tolist(), tokenizer)
    test_ds  = BanMANIDataset(te["text_sep_token"].tolist(),
                               te["label"].tolist(), tokenizer)

    args = TrainingArguments(
        output_dir=f"./output_banglabert_seed{seed}",
        num_train_epochs=epochs,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=4,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_steps=20,
        report_to="none",
        seed=seed,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
    )

    trainer.train()
    results = trainer.evaluate()
    out = {k: round(v, 4) for k, v in results.items() if "runtime" not in k}
    print(f"\\nSEED {seed} FINAL RESULTS: {json.dumps(out, indent=2)}")

    return {
        "seed": seed,
        "train_mani_pct": round(float(tr["label"].mean()*100), 1),
        "test_mani_pct": round(float(te["label"].mean()*100), 1),
        **out
    }

if __name__ == "__main__":
    seed = int(sys.argv[1]) if len(sys.argv) > 1 else 0
    csv  = sys.argv[2] if len(sys.argv) > 2 else "/kaggle/working/BanMANI.csv"
    result = run_seed(csv, seed)

    out_path = "/kaggle/working/banglabert_restratified.json"
    try:
        with open(out_path) as f:
            all_results = json.load(f)
    except:
        all_results = []
    all_results.append(result)
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\\nSaved to {out_path}")
'''

with open('/kaggle/working/banglabert_restrat.py', 'w') as f:
    f.write(code)
print("Script written successfully")

Script written successfully


In [11]:
!python /kaggle/working/banglabert_restrat.py 0 /kaggle/working/BanMANI.csv

  File "/kaggle/working/banglabert_restrat.py", line 35
    print(f"\n{\'=\'*55}")
                ^
SyntaxError: unexpected character after line continuation character


In [12]:
script = """import sys, os, json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
sys.path.insert(0, "/kaggle/working")
from data.preprocess import load_dataset
import torch
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

class BanMANIDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.encodings = tokenizer(texts, max_length=max_len, truncation=True, padding=False)
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_mani": f1_score(labels, preds, pos_label=1, zero_division=0),
    }

def run_seed(csv_path, seed, model_name="sagorsarker/bangla-bert-base", epochs=5):
    print("=" * 55)
    print("BanglaBERT | Re-stratified split | Seed " + str(seed))
    print("=" * 55)

    train_df, test_df = load_dataset(csv_path)
    combined = pd.concat([train_df, test_df], ignore_index=True)
    tr, te = train_test_split(combined, test_size=150, stratify=combined["label"], random_state=seed)
    tr = tr.reset_index(drop=True)
    te = te.reset_index(drop=True)

    print("Train: " + str(len(tr)) + " samples (" + str(round(tr["label"].mean()*100, 1)) + "% MANI)")
    print("Test:  " + str(len(te)) + " samples (" + str(round(te["label"].mean()*100, 1)) + "% MANI)")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, ignore_mismatched_sizes=True)

    train_ds = BanMANIDataset(tr["text_sep_token"].tolist(), tr["label"].tolist(), tokenizer)
    test_ds  = BanMANIDataset(te["text_sep_token"].tolist(), te["label"].tolist(), tokenizer)

    args = TrainingArguments(
        output_dir="./output_banglabert_seed" + str(seed),
        num_train_epochs=epochs,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=4,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_steps=20,
        report_to="none",
        seed=seed,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
    )

    trainer.train()
    results = trainer.evaluate()
    out = {k: round(v, 4) for k, v in results.items() if "runtime" not in k}
    print("SEED " + str(seed) + " FINAL RESULTS:")
    print(json.dumps(out, indent=2))

    return {
        "seed": seed,
        "train_mani_pct": round(float(tr["label"].mean()*100), 1),
        "test_mani_pct": round(float(te["label"].mean()*100), 1),
        **out
    }

if __name__ == "__main__":
    seed = int(sys.argv[1]) if len(sys.argv) > 1 else 0
    csv  = sys.argv[2] if len(sys.argv) > 2 else "/kaggle/working/BanMANI.csv"
    result = run_seed(csv, seed)

    out_path = "/kaggle/working/banglabert_restratified.json"
    try:
        with open(out_path) as f:
            all_results = json.load(f)
    except:
        all_results = []
    all_results.append(result)
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print("Saved to " + out_path)
"""

with open('/kaggle/working/banglabert_restrat.py', 'w') as f:
    f.write(script)
print("Script written successfully")

Script written successfully


In [13]:
!python /kaggle/working/banglabert_restrat.py 0 /kaggle/working/BanMANI.csv

2026-06-15 04:09:17.466774: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781496557.649507     166 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781496557.708294     166 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781496558.147767     166 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781496558.147816     166 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781496558.147820     166 computation_placer.cc:177] computation placer alr

In [14]:
!python /kaggle/working/banglabert_restrat.py 1 /kaggle/working/BanMANI.csv

2026-06-15 04:15:59.533438: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781496959.556523    1187 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781496959.564183    1187 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781496959.583292    1187 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781496959.583325    1187 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781496959.583328    1187 computation_placer.cc:177] computation placer alr

In [16]:
!python /kaggle/working/banglabert_restrat.py 2 /kaggle/working/BanMANI.csv

2026-06-15 04:33:59.677182: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781498039.708839    1821 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781498039.721209    1821 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781498039.748297    1821 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781498039.748336    1821 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781498039.748340    1821 computation_placer.cc:177] computation placer alr

In [17]:
import shutil, os

for seed in [0, 1, 2]:
    path = f'/kaggle/working/output_banglabert_seed{seed}'
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"Deleted {path}")

print("Space freed")
print("Working dir:", os.listdir('/kaggle/working/'))

Deleted /kaggle/working/output_banglabert_seed0
Deleted /kaggle/working/output_banglabert_seed1
Deleted /kaggle/working/output_banglabert_seed2
Space freed
Working dir: ['.virtual_documents', 'banglabert_restrat.py', 'banglabert_restratified.json', 'BanMANI.csv', 'data']


In [18]:
!python /kaggle/working/banglabert_restrat.py 2 /kaggle/working/BanMANI.csv


2026-06-15 04:37:00.462341: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781498220.485150    1954 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781498220.493239    1954 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781498220.513053    1954 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781498220.513084    1954 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781498220.513087    1954 computation_placer.cc:177] computation placer alr

In [19]:
import json, numpy as np

with open('/kaggle/working/banglabert_restratified.json') as f:
    results = json.load(f)

print("=" * 60)
print("  BANGLABERT RE-STRATIFIED RESULTS")
print("=" * 60)
print(f"{'Seed':<6} {'Train MANI%':<14} {'Test MANI%':<12} {'F1 Macro':<12} {'F1 MANI'}")
print("-" * 60)
for r in results:
    print(f"{r['seed']:<6} {r['train_mani_pct']:<14} {r['test_mani_pct']:<12} "
          f"{r.get('eval_f1_macro', '?'):<12} {r.get('eval_f1_mani', '?')}")

f1s = [r.get('eval_f1_macro', 0) for r in results]
f1_manis = [r.get('eval_f1_mani', 0) for r in results]
mean_f1 = np.mean(f1s)
std_f1 = np.std(f1s)
ci = 1.96 * std_f1 / np.sqrt(len(f1s))

mean_mani = np.mean(f1_manis)

print()
print("SUMMARY:")
print(f"Original BanglaBERT (official split):       F1 Macro = 0.4258  | F1 MANI = 0.5789")
print(f"Re-stratified BanglaBERT (mean +/- 95%CI): F1 Macro = {mean_f1:.4f} +/- {ci:.4f} | F1 MANI = {mean_mani:.4f}")
print(f"Difference (F1 Macro): {mean_f1 - 0.4258:+.4f}")
print(f"Difference (F1 MANI):  {mean_mani - 0.5789:+.4f}")

if mean_f1 - 0.4258 > ci:
    print("\n=> Distribution shift DID hurt BanglaBERT significantly")
    print("   True transformer capability is higher than originally reported")
else:
    print("\n=> Distribution shift did NOT significantly affect BanglaBERT")
    print("   Classical ML is genuinely more robust to this shift")

print()
print("COMPARISON TO TFIDF:")
print(f"TF-IDF (original split):       F1 Macro = 0.5717")
print(f"TF-IDF (re-stratified, mean):  F1 Macro = 0.6859")
print(f"BanglaBERT (re-stratified):    F1 Macro = {mean_f1:.4f}")
if mean_f1 > 0.5717:
    print(f"\n=> Under matched distributions, BanglaBERT BEATS the original TF-IDF baseline")
if mean_f1 > 0.6859:
    print(f"=> Under matched distributions, BanglaBERT also BEATS re-stratified TF-IDF")
else:
    print(f"=> TF-IDF re-stratified ({0.6859:.4f}) still beats BanglaBERT re-stratified ({mean_f1:.4f})")

  BANGLABERT RE-STRATIFIED RESULTS
Seed   Train MANI%    Test MANI%   F1 Macro     F1 MANI
------------------------------------------------------------
0      66.5           66.7         0.655        0.77
1      66.5           66.7         0.6362       0.7963
2      66.5           66.7         0.5417       0.8167

SUMMARY:
Original BanglaBERT (official split):       F1 Macro = 0.4258  | F1 MANI = 0.5789
Re-stratified BanglaBERT (mean +/- 95%CI): F1 Macro = 0.6110 +/- 0.0561 | F1 MANI = 0.7943
Difference (F1 Macro): +0.1852
Difference (F1 MANI):  +0.2154

=> Distribution shift DID hurt BanglaBERT significantly
   True transformer capability is higher than originally reported

COMPARISON TO TFIDF:
TF-IDF (original split):       F1 Macro = 0.5717
TF-IDF (re-stratified, mean):  F1 Macro = 0.6859
BanglaBERT (re-stratified):    F1 Macro = 0.6110

=> Under matched distributions, BanglaBERT BEATS the original TF-IDF baseline
=> TF-IDF re-stratified (0.6859) still beats BanglaBERT re-stratified